In [ ]:
!pip uninstall -y transformers peft accelerate -q
!pip install transformers==4.40.0 accelerate==0.30.1 datasets scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import re
import os
os.environ["WANDB_DISABLED"] = "true"

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("IMDB Dataset.csv")   # make sure name matches
df.head()

In [ ]:
df.columns = ['text', 'label']
df['label'] = df['label'].map({'positive':1, 'negative':0})
df.dropna(inplace=True)

# 🔥 reduce dataset for speed
df = df.sample(4000, random_state=42)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

In [ ]:
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
val_dataset = Dataset.from_dict({'text': val_texts.tolist(), 'label': val_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_dataset.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_dataset.set_format(type='torch', columns=['input_ids','attention_mask','label'])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,   # 🔥 fast
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_dir='./logs',
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(test_labels, preds)
print("Confusion Matrix:\n", cm)

The BERT model was successfully fine-tuned on the dataset.
The model achieved good accuracy and F1-score, showing effective text classification.
Using GPU and reduced dataset helped complete training quickly.
The confusion matrix shows most predictions are correct.

In [17]:
import nbformat
import os
from google.colab import files

# 🔹 find notebook file automatically
notebook_files = [f for f in os.listdir() if f.endswith(".ipynb")]

print("Found notebook:", notebook_files)

# pick first notebook
notebook_name = notebook_files[0]

# 🔹 load notebook
nb = nbformat.read(notebook_name, as_version=4)

# 🔹 remove widget metadata
nb.metadata.pop("widgets", None)

# 🔹 clean each cell
for cell in nb.cells:
    if "metadata" in cell:
        cell["metadata"].pop("widgets", None)

# 🔹 save cleaned notebook
clean_name = "FINAL_Submission.ipynb"
nbformat.write(nb, clean_name)

print("✅ Cleaned notebook saved as:", clean_name)

# 🔹 download
files.download(clean_name)

Found notebook: []


IndexError: list index out of range